In [2]:
"""
Download weekly USD/X exchange rates for all sovereign CDS countries.
Convention: each column = units of local currency per 1 USD (USD/XXX).

Run locally:  python download_fx_rates.py

Output:  data/processed/FX/Weekly_FX_Rates.csv
         (wide format: Date index, one column per CDS country name)

Requires:  pip install yfinance pandas
"""

import yfinance as yf
import pandas as pd
import time
from pathlib import Path

# ═══════════════════════════════════════════════════════════════════════
# COUNTRY → CURRENCY MAPPING
# ═══════════════════════════════════════════════════════════════════════
# Key   = country name (must match your CDS CSV column names exactly)
# Value = ISO 4217 currency code
#
# Countries sharing a currency (e.g., EUR, AED) each get their own
# column in the output — the FX series is simply duplicated.
# ═══════════════════════════════════════════════════════════════════════

COUNTRY_CURRENCY = {
    # ── Eurozone (all map to EUR) ─────────────────────────────────────
    'Austria':        'EUR',
    'Belgium':        'EUR',
    'Bulgaria':       'EUR',  # Note: Bulgaria uses BGN, but if your CDS
                              # file has Bulgaria, change to 'BGN'
    'Croatia':        'EUR',  # joined EUR 2023; was HRK before
    'Cyprus':         'EUR',
    'Estonia':        'EUR',
    'Finland':        'EUR',
    'France':         'EUR',
    'Germany':        'EUR',
    'Greece':         'EUR',
    'Ireland':        'EUR',
    'Italy':          'EUR',
    'Latvia':         'EUR',
    'Lithuania':      'EUR',
    'Netherlands':    'EUR',
    'Portugal':       'EUR',
    'Slovakia':       'EUR',
    'Slovenia':       'EUR',
    'Spain':          'EUR',

    # ── Gulf states ───────────────────────────────────────────────────
    'Abu Dhabi':      'AED',
    'Dubai':          'AED',
    'Bahrain':        'BHD',
    'Kuwait':         'KWD',  # Yahoo: USDKWD=X — may have gaps
    'Oman':           'OMR',
    'Qatar':          'QAR',
    'Saudi Arabia':   'SAR',

    # ── Americas ──────────────────────────────────────────────────────
    'Argentina':      'ARS',
    'Brazil':         'BRL',
    'Chile':          'CLP',
    'Colombia':       'COP',
    'Dominican Republic': 'DOP',
    'Ecuador':        'USD',  # dollarized
    'El Salvador':    'USD',  # dollarized
    'Guatemala':      'GTQ',
    'Jamaica':        'JMD',
    'Mexico':         'MXN',
    'Panama':         'USD',  # dollarized
    'Paraguay':       'PYG',
    'Peru':           'PEN',
    'Trinidad and Tobago': 'TTD',
    'United States':  'USD',
    'Uruguay':        'UYU',
    'Venezuela':      'VES',

    # ── Europe (non-EUR) ──────────────────────────────────────────────
    'Czech Republic': 'CZK',
    'Denmark':        'DKK',
    'Hungary':        'HUF',
    'Iceland':        'ISK',
    'Norway':         'NOK',
    'Poland':         'PLN',
    'Romania':        'RON',
    'Russia':         'RUB',
    'Serbia':         'RSD',
    'Sweden':         'SEK',
    'Switzerland':    'CHF',
    'Turkey':         'TRY',
    'Ukraine':        'UAH',
    'United Kingdom': 'GBP',

    # ── Asia-Pacific ──────────────────────────────────────────────────
    'Australia':      'AUD',
    'China':          'CNY',
    'Hong Kong':      'HKD',
    'India':          'INR',
    'Indonesia':      'IDR',
    'Japan':          'JPY',
    'Kazakhstan':     'KZT',
    'Malaysia':       'MYR',
    'Mongolia':       'MNT',
    'New Zealand':    'NZD',
    'Pakistan':       'PKR',
    'Philippines':    'PHP',
    'Singapore':      'SGD',
    'South Korea':    'KRW',
    'Sri Lanka':      'LKR',
    'Taiwan':         'TWD',
    'Thailand':       'THB',
    'Vietnam':        'VND',

    # ── Africa & Middle East ──────────────────────────────────────────
    'Egypt':          'EGP',
    'Ghana':          'GHS',
    'Iraq':           'IQD',
    'Israel':         'ILS',
    'Jordan':         'JOD',
    'Kenya':          'KES',
    'Lebanon':        'LBP',
    'Morocco':        'MAD',
    'Nigeria':        'NGN',
    'South Africa':   'ZAR',
    'Tunisia':        'TND',
}

# ═══════════════════════════════════════════════════════════════════════
# DOWNLOAD SETTINGS
# ═══════════════════════════════════════════════════════════════════════

START = '2000-01-01'
END   = '2025-01-01'

# ═══════════════════════════════════════════════════════════════════════
# BUILD UNIQUE CURRENCY LIST & DOWNLOAD
# ═══════════════════════════════════════════════════════════════════════

unique_currencies = sorted(set(COUNTRY_CURRENCY.values()))
print(f"Countries mapped : {len(COUNTRY_CURRENCY)}")
print(f"Unique currencies: {len(unique_currencies)}")
print(f"Currencies: {', '.join(unique_currencies)}")

# Build yfinance tickers (skip USD — rate is always 1.0)
tickers_to_download = [f"USD{ccy}=X" for ccy in unique_currencies if ccy != 'USD']

print(f"\nDownloading {len(tickers_to_download)} FX pairs from Yahoo Finance...")
print(f"Period: {START} → {END}, interval: 1d\n")

# Download in batches to avoid throttling
BATCH_SIZE = 20
all_data = {}

for i in range(0, len(tickers_to_download), BATCH_SIZE):
    batch = tickers_to_download[i:i+BATCH_SIZE]
    batch_str = ' '.join(batch)
    print(f"  Batch {i//BATCH_SIZE + 1}: {batch[0]} ... {batch[-1]}")

    try:
        df = yf.download(
            batch_str,
            start=START,
            end=END,
            interval='1d',
            auto_adjust=True,
            progress=False
        )

        # yfinance returns MultiIndex columns when multiple tickers
        if len(batch) == 1:
            # Single ticker → flat columns
            ticker = batch[0]
            if not df.empty:
                all_data[ticker] = df['Close']
        else:
            # Multiple tickers → (Price, Ticker) MultiIndex
            if 'Close' in df.columns.get_level_values(0):
                close = df['Close']
                for ticker in batch:
                    if ticker in close.columns:
                        all_data[ticker] = close[ticker]
                    else:
                        print(f"    ⚠ {ticker} — no data returned")
            else:
                print(f"    ⚠ Batch returned unexpected format")

    except Exception as e:
        print(f"    ✗ Batch error: {e}")

    # Be polite to Yahoo
    if i + BATCH_SIZE < len(tickers_to_download):
        time.sleep(2)

# ═══════════════════════════════════════════════════════════════════════
# ASSEMBLE INTO WIDE DATAFRAME (one column per COUNTRY)
# ═══════════════════════════════════════════════════════════════════════

# First, build currency-level DataFrame
fx_by_ccy = pd.DataFrame(all_data)
fx_by_ccy.index.name = 'Date'

# Rename columns from "USDXXX=X" → "XXX"
fx_by_ccy.columns = [col.replace('USD', '').replace('=X', '') for col in fx_by_ccy.columns]

# Add USD column (always 1.0)
fx_by_ccy['USD'] = 1.0

print(f"\nCurrency-level data: {fx_by_ccy.shape[0]} weeks × {fx_by_ccy.shape[1]} currencies")

# Now map to country names
fx_by_country = pd.DataFrame(index=fx_by_ccy.index)

missing = []
for country, ccy in COUNTRY_CURRENCY.items():
    if ccy in fx_by_ccy.columns:
        fx_by_country[country] = fx_by_ccy[ccy]
    else:
        missing.append((country, ccy))
        print(f"  ⚠ Missing currency data for {country} ({ccy})")

if missing:
    print(f"\n⚠ {len(missing)} countries missing FX data (may need manual download)")

# ═══════════════════════════════════════════════════════════════════════
# FORWARD-FILL AND CLEAN
# ═══════════════════════════════════════════════════════════════════════

# Forward fill small gaps (weekends, holidays already handled by weekly)
fx_by_country = fx_by_country.ffill()

# Report coverage
print(f"\n{'Country':<25} {'Currency':>5}  {'First':>12} {'Last':>12}  {'Gaps':>5}")
print("-" * 70)
for country in sorted(fx_by_country.columns):
    col = fx_by_country[country].dropna()
    if len(col) == 0:
        print(f"{country:<25} {'—':>5}  {'NO DATA':>12} {'':>12}  {'':>5}")
        continue
    gaps = fx_by_country[country].isna().sum()
    ccy  = COUNTRY_CURRENCY[country]
    print(f"{country:<25} {ccy:>5}  {col.index[0].strftime('%Y-%m-%d'):>12} "
          f"{col.index[-1].strftime('%Y-%m-%d'):>12}  {gaps:>5}")

# ═══════════════════════════════════════════════════════════════════════
# SAVE
# ═══════════════════════════════════════════════════════════════════════

out_dir = Path('data/processed/Macroeconomic_variables')
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / 'Daily_FX_Rates.csv'
fx_by_country.to_csv(out_path)
print(f"\n✓ Saved: {out_path}")
print(f"  Shape: {fx_by_country.shape[0]} days × {fx_by_country.shape[1]} countries")
print(f"  Convention: USD/XXX (units of local currency per 1 USD)")

# Also save a currency-level version for reference
ccy_path = out_dir / 'Daily_FX_Rates_by_currency.csv'
fx_by_ccy.to_csv(ccy_path)
print(f"  Also saved currency-level file: {ccy_path}")

Countries mapped : 86
Unique currencies: 64
Currencies: AED, ARS, AUD, BHD, BRL, CHF, CLP, CNY, COP, CZK, DKK, DOP, EGP, EUR, GBP, GHS, GTQ, HKD, HUF, IDR, ILS, INR, IQD, ISK, JMD, JOD, JPY, KES, KRW, KWD, KZT, LBP, LKR, MAD, MNT, MXN, MYR, NGN, NOK, NZD, OMR, PEN, PHP, PKR, PLN, PYG, QAR, RON, RSD, RUB, SAR, SEK, SGD, THB, TND, TRY, TTD, TWD, UAH, USD, UYU, VES, VND, ZAR

Period: 2000-01-01 → 2025-01-01, interval: 1d

  Batch 1: USDAED=X ... USDIDR=X
  Batch 2: USDILS=X ... USDNZD=X


$USDMNT=X: possibly delisted; no price data found  (1d 2000-01-01 -> 2025-01-01)

1 Failed download:
['USDMNT=X']: possibly delisted; no price data found  (1d 2000-01-01 -> 2025-01-01)


  Batch 3: USDOMR=X ... USDUYU=X
  Batch 4: USDVES=X ... USDZAR=X

Currency-level data: 6522 weeks × 64 currencies

Country                   Currency         First         Last   Gaps
----------------------------------------------------------------------
Abu Dhabi                   AED    2003-12-01   2024-12-31   1020
Argentina                   ARS    2001-07-13   2024-12-31    399
Australia                   AUD    2006-05-16   2024-12-31   1661
Austria                     EUR    2003-12-01   2024-12-31   1020
Bahrain                     BHD    2002-03-22   2024-12-31    579
Belgium                     EUR    2003-12-01   2024-12-31   1020
Brazil                      BRL    2003-12-01   2024-12-31   1020
Bulgaria                    EUR    2003-12-01   2024-12-31   1020
Chile                       CLP    2003-12-01   2024-12-31   1020
China                       CNY    2001-06-25   2024-12-31    385
Colombia                    COP    2003-01-02   2024-12-31    783
Croatia           